## Initializing OCR Reader

In [1]:
import easyocr

reader = easyocr.Reader(['en'], gpu=True)  



Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


## Preprocessing the Images

In [8]:
import cv2
import numpy as np

def preprocess_for_ocr(image_path):
    img = cv2.imread(image_path)

    if img is None:
        raise ValueError(f"Could not read image {image_path}")

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Increase contrast
    gray = cv2.normalize(gray, None, alpha=0, beta=255,
                          norm_type=cv2.NORM_MINMAX)

    # Adaptive thresholding (very important for ads)
    thresh = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11,
        2
    )

    return thresh


## Running OCR for one Image

In [10]:
def run_ocr_on_image(image_path, reader, conf_threshold=0.4):
    processed_img = preprocess_for_ocr(image_path)
    results = reader.readtext(processed_img)
    texts = []

    for bbox, text, conf in results:
        if conf >= conf_threshold:
            texts.append(text)

    return texts



## OCR Runners for one Video

In [11]:
import os
import json

def run_ocr_for_video_folder(video_folder_path, output_json, reader, conf_threshold=0.4):
    keyframes_dir = os.path.join(video_folder_path, "keyframes")

    if not os.path.isdir(keyframes_dir):
        print(f"[WARN] No keyframes in {video_folder_path}")
        return

    ocr_results = {}

    for fname in sorted(os.listdir(keyframes_dir)):
        if not fname.lower().endswith(".jpg"):
            continue

        # expects: shot_001.jpg
        try:
            shot_id = int(fname.split("_")[1].split(".")[0])
        except:
            print(f"[WARN] Bad filename: {fname}")
            continue

        img_path = os.path.join(keyframes_dir, fname)
        texts = run_ocr_on_image(img_path, reader, conf_threshold)

        ocr_results[shot_id] = texts

    os.makedirs(os.path.dirname(output_json), exist_ok=True)

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(ocr_results, f, indent=2)


## Batch OCR over all the Vodeos

In [13]:
SEGMENTATION_DIR = "../data/segmentation"
OCR_DIR = "../data/ocr"

os.makedirs(OCR_DIR, exist_ok=True)

for video_name in os.listdir(SEGMENTATION_DIR):
    video_folder = os.path.join(SEGMENTATION_DIR, video_name)

    if not os.path.isdir(video_folder):
        continue

    out_json = os.path.join(OCR_DIR, f"{video_name}.json")

    if os.path.exists(out_json):
        print(f"[SKIP] OCR exists for {video_name}")
        continue

    print(f"[OCR] Processing {video_name}")
    run_ocr_for_video_folder(
        video_folder,
        out_json,
        reader,
        conf_threshold=0.3
    )

print("OCR extraction completed for all videos")


[OCR] Processing 16 0613 PROJ COLOR CHEF
[OCR] Processing 6607010_2158732_DE_de_9000D_Winter Sale_Feed&Masthead_YT_Women_Step2_16x9_20s_D - H&M (720p, h264, youtube)
[OCR] Processing Adidas _Beyond The Blue_ AI Commercial (Midjourney + @RunwayML + @topazlabs )  AI Advertising
[OCR] Processing ad_01
[OCR] Processing All the Best Moments are Better With Pepsi
[OCR] Processing Banned M&M's Commercial
[OCR] Processing BORZ WEAR (Streetwear Commercial Video) - ICE SQUAD MEDIA
[OCR] Processing CINEMATIC PRODUCT VIDEO _ Go Good Drinks
[OCR] Processing Clothing commercial video _ Jacferdi _ Fujifilm X-T3
[OCR] Processing Essentials Clothing Shoot _ Promotional Video
[OCR] Processing Every Table Has A Story
[OCR] Processing Gucci Presents_ Gucci Oud, the new unisex fragrance
[OCR] Processing Introducting ACQUA DI GIÒ PROFONDO PARFUM by Giorgio Armani


ValueError: Could not read image ../data/segmentation\Introducting ACQUA DI GIÒ PROFONDO PARFUM by Giorgio Armani\keyframes\shot_001.jpg